# RNN, gated recurrence and attention — live companion (90 min)

This notebook is designed to be **coded live with the class** while following the final RNN slides.  
It deliberately avoids long training runs and external datasets: every execution point should complete in seconds on CPU.

**Reference notebooks (extended experiments):**
- [Géron / course mirror — Ch. 13](https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/13_processing_sequences_using_rnns_and_cnns.ipynb)
- [Géron / course mirror — Ch. 14](https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/14_nlp_with_rnns_and_attention.ipynb)



## 0 — Setup

**Slides:** tensor conventions used throughout the deck.  
We will use `batch_first=True`, hence sequence tensors are `(N, T, D)`.


In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)
print('PyTorch:', torch.__version__)

def show_shape(name, x):
    print(f'{name:>10s}: {tuple(x.shape)}')


## 1 — Vanilla RNN: one cell, repeated computation 

**Slides 8–15. Key points**
- the *same* parameters are reused at every timestep;
- unrolling creates repeated computation, not new parameters;
- with `batch_first=True`, `output` is `(N,T,D_h)` but `h_n` is `(layers·directions,N,D_h)`;
- for a 1-layer unidirectional RNN, `output[:, -1] == h_n[0]`.

We verify this by reproducing `nn.RNN` manually using its own weights.


In [ ]:
N, T, D_x, D_h = 2, 4, 3, 5
x = torch.randn(N, T, D_x)
rnn = nn.RNN(D_x, D_h, batch_first=True, nonlinearity='tanh')

# TODO 1: run the RNN and unpack sequence output and final state.
output, h_n = ...
show_shape('x', x)
show_shape('output', output)
show_shape('h_n', h_n)

# TODO 2: manually unroll the SAME recurrent cell using rnn.weight_* and rnn.bias_*.
h = torch.zeros(N, D_h)
manual = []
for t in range(T):
    h = ...
    manual.append(h)
manual_output = torch.stack(manual, dim=1)

assert manual_output.shape == output.shape
print('manual unroll == nn.RNN:', torch.allclose(manual_output, output, atol=1e-6))


**Checkpoint.** Where is parameter sharing visible in the code? What would have to change if each timestep had different parameters?


## 2 — BPTT: gradient dynamics, clipping and truncation 

**Slides 21–28. Key points**
- long-range gradients contain products of local derivatives;
- state dynamics and gradient dynamics are different questions;
- clipping controls *exploding update magnitude*;
- truncated BPTT may carry the numerical state while detaching the graph.

First, use the simplest linear recurrence so the gradient product is visible without other effects.


In [ ]:
# TODO: verify dh_T/dh_0 = alpha**T for a linear recurrence.
for alpha in (0.8, 1.0, 1.2):
    vals = []
    for steps in (5, 10, 20):
        h0 = torch.tensor(1.0, requires_grad=True)
        h = h0
        for _ in range(steps):
            h = ...
        h.backward()
        vals.append(float(h0.grad))
    print(f'alpha={alpha}: gradients {vals}')


In [ ]:
model = nn.RNN(3, 8, batch_first=True)
head = nn.Linear(8, 1)
x_long = 20 * torch.randn(16, 40, 3)
y = head(model(x_long)[0][:, -1]).pow(2).mean()
y.backward()

# TODO: clip the global gradient norm to 1.0.
returned_norm = ...
print('pre-clip norm returned by clip_grad_norm_:', returned_norm)

# TODO: carry the numerical hidden state but cut its graph history.
h = torch.randn(1, 1, 8, requires_grad=True)
chunk = torch.randn(1, 5, 3)
_, h_next = model(chunk, h)
h_next = ...
assert h_next.grad_fn is None


**Checkpoint.** `detach()` does **not** reset the hidden-state values. It only changes which earlier operations receive gradients.


## 3 — LSTM: gates plus an additive cell-state path 

**Slides 31–40. Key points**
- gates are vectors in `(0,1)`, not Boolean switches;
- the LSTM carries both `h_t` and `c_t`;
- `c_t` is updated additively after gating;
- `batch_first=True` changes sequence tensor layout, **not** `h_n/c_n` layout.

We first compute one LSTM update explicitly, then inspect `nn.LSTM`.


In [ ]:
N, D_x, D_h = 2, 3, 4
x_t = torch.randn(N, D_x)
h_prev = torch.randn(N, D_h)
c_prev = torch.randn(N, D_h)
W_x = torch.randn(4 * D_h, D_x) / math.sqrt(D_x)
W_h = torch.randn(4 * D_h, D_h) / math.sqrt(D_h)
b = torch.zeros(4 * D_h)

gates = x_t @ W_x.T + h_prev @ W_h.T + b
i_pre, f_pre, g_pre, o_pre = gates.chunk(4, dim=-1)

# TODO: sigmoid gates, tanh candidate, additive cell update, exposed hidden state.
i_t = ...
f_t = ...
g_t = ...
o_t = ...
c_t = ...
h_t = ...

show_shape('c_t', c_t)
show_shape('h_t', h_t)


In [ ]:
N, T, D_x, D_h = 3, 6, 4, 5
x = torch.randn(N, T, D_x)

# TODO: create a batch-first LSTM and inspect output, h_n and c_n.
lstm = ...
output, (h_n, c_n) = ...
show_shape('output', output)
show_shape('h_n', h_n)
show_shape('c_n', c_n)


**Checkpoint.** Why can `c_t` and `h_t` carry different information even though they have the same dimensionality?


## 4 — GRU: gated interpolation with one recurrent state 

**Slides 41–44. Key points**
- reset gate affects candidate construction;
- update gate interpolates old and candidate state;
- unlike LSTM, GRU has no separate `c_t`.


In [ ]:
h_prev = torch.tensor([[1.0, -1.0, 0.5]])
h_tilde = torch.tensor([[-2.0, 2.0, 1.5]])

# TODO: implement h_t = z*h_prev + (1-z)*h_tilde.
for z in (0.0, 0.25, 0.75, 1.0):
    z_t = torch.full_like(h_prev, z)
    h_t = ...
    print(f'z={z:>4}: h_t={h_t.tolist()}')

# TODO: instantiate nn.GRU with batch_first=True and inspect its two outputs.
x = torch.randn(3, 6, 4)
gru = ...
output, h_n = ...
show_shape('output', output)
show_shape('h_n', h_n)


**Checkpoint.** Under the convention used in the slides, what does `z_t ≈ 1` do? Why is this convention worth checking when reading another text/library?


## 5 — Layer depth and direction are separate axes

**Slides 45–49. Key points**
- stacking adds a layer axis to state metadata;
- bidirectionality doubles sequence features and direction metadata;
- bidirectional processing is not strict causal streaming.


In [ ]:
N, T, D_x, D_h = 2, 7, 3, 4
x = torch.randn(N, T, D_x)

# TODO: 2-layer bidirectional LSTM.
bi = ...
out, (h_n, c_n) = bi(x)
show_shape('out', out)
show_shape('h_n', h_n)

# TODO: split forward/backward features and reshape h_n to
# (layers, directions, N, D_h).
forward_features = ...
backward_features = ...
h_view = ...
show_shape('forward', forward_features)
show_shape('backward', backward_features)
show_shape('h view', h_view)


## 6 — Seq2seq training: teacher forcing and perplexity 

**Slides 29, 50–55. Key points**
- teacher forcing is a **training input strategy**;
- decoder inputs are shifted targets: `<BOS>, y_1, ..., y_{T-1}`;
- inference feeds model-generated tokens instead;
- token-level cross-entropy and perplexity are directly related: `PPL = exp(mean NLL)`.


In [ ]:
target = torch.tensor([[4, 7, 2, 9], [6, 3, 5, 8]])
bos = torch.ones((target.size(0), 1), dtype=torch.long)

# TODO 1: teacher-forced decoder input = BOS + target without last token.
decoder_in = ...
print('target:
', target)
print('decoder input:
', decoder_in)

# TODO 2: compute cross-entropy and PPL for logits biased toward the correct token.
V = 10
logits = torch.zeros(*target.shape, V)
logits.scatter_(-1, target.unsqueeze(-1), 2.0)
loss = ...
ppl = ...
print('loss=', float(loss), 'PPL=', float(ppl))


**Checkpoint.** If we change the tokenizer/vocabulary, can we compare perplexity values naively? Why not?


## 7 — Attention and self-attention: score → softmax → weighted sum 

**Slides 58–68. Key points**
- a query selects among keys and aggregates values;
- attention weights form a normalized soft alignment;
- self-attention uses Q/K/V from the same sequence;
- without positional information, self-attention is permutation equivariant rather than sequence-order aware.


In [ ]:
D = 4
keys = torch.eye(D)
values = torch.tensor([[10., 0.], [0., 10.], [5., 5.], [-5., 5.]])
query = torch.tensor([0.1, 2.0, 0.2, -0.1])

# TODO: scaled dot-product scores, softmax over source positions, weighted context.
scores = ...
alpha = ...
context = ...
print('scores:', scores)
print('weights:', alpha, 'sum=', alpha.sum())
print('context:', context)


In [ ]:
torch.manual_seed(2)
T, D, d_k = 5, 6, 4
X = torch.randn(T, D)
Wq, Wk, Wv = [torch.randn(D, d_k) / math.sqrt(D) for _ in range(3)]

def self_attention(X):
    # TODO: return Y and attention matrix A.
    Q = ...
    K = ...
    V = ...
    A = ...
    Y = ...
    return Y, A

Y, A = self_attention(X)
perm = torch.tensor([2, 0, 4, 1, 3])
Y_perm, _ = self_attention(X[perm])
print('Permutation equivariance without position:',
      torch.allclose(Y_perm, Y[perm], atol=1e-6))


### Final retrieval questions
1. Why does parameter sharing reduce model size but not temporal computation depth?  
2. What is the difference between carrying a state and carrying a gradient history?  
3. LSTM: why are `c_t` and `h_t` separate? GRU: what replaces that separation?  
4. Why is teacher forcing a training strategy rather than an inference algorithm?  
5. In attention, along which axis must the softmax normalize?  
6. Why does self-attention need positional information if order matters?


# Part II — Homework: recurrent memory + time-series forecasting

**Continue in this same notebook. Do not start a second file.**

- **Expected effort:** about **4 h 30 min per team**
- **Team size:** **1–3 students**
- Submit **one executed notebook**.
- Short written answers should be concise and evidence-based.
- Training must remain lightweight enough to run on Colab CPU/GPU.

The homework has two complementary threads:

1. **Long-range sequence classification**  
   `vanilla RNN → gated recurrence → attention`

2. **Time-series forecasting**  
   `stationarity → windowing → many-to-many next-step training → autoregressive rollout`

## Time budget

| Activity | Target |
|---|---:|
| A. Variable length, padding and masking | 20 min |
| B. Vanilla RNN long-range baseline | 30 min |
| C. GRU/LSTM comparison | 30 min |
| D. Attention pooling + self-attention order probe | 35 min |
| E. Time-series exploration and stationarity | 40 min |
| F. Many-to-many next-step forecasting | 45 min |
| G. Recursive multi-step forecasting | 40 min |
| H. Final comparison and interpretation | 30 min |
| **Total** | **270 min = 4 h 30 min** |

> **Important:** a recurrent model does not mathematically require a stationary input series.  
> Here stationarity is studied as a modeling/diagnostic issue: changing level, trend, scale and distribution can affect optimization and generalization.


In [ ]:
# Homework setup
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

torch.manual_seed(11)
np.random.seed(11)
random.seed(11)

# Avoid excessive CPU thread overhead on the small recurrent workloads used here.
torch.set_num_threads(1)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## A — Variable length, padding and masking
We use a deliberately controlled long-range classification task.

For each sequence:

- the **first token** is `1` or `2`;
- the middle tokens are distractors from `3..9`;
- the last token is another independent `1/2` distractor;
- sequence length is variable;
- padding token is `0`;
- the label depends **only on the first token**:

\[
y = x_1 - 1.
\]

A final-state classifier must therefore preserve information from the beginning of the sequence across many irrelevant timesteps.

### Tasks

1. Inspect one batch and report the shapes of `X`, `lengths`, `mask`, `y`.
2. Verify that `mask == False` exactly identifies padded positions.
3. Print two sequences before and after removing padding.
4. Explain in **3–5 sentences** why padding is a batching device, not a semantic timestep.


In [ ]:
class LongRangeFirstToken(Dataset):
    def __init__(self, n=3000, min_len=10, max_len=30, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.samples = []
        for _ in range(n):
            L = int(torch.randint(min_len, max_len + 1, (1,), generator=g))
            x = torch.randint(3, 10, (L,), generator=g)
            first = int(torch.randint(1, 3, (1,), generator=g))
            last = int(torch.randint(1, 3, (1,), generator=g))
            x[0], x[-1] = first, last
            y = first - 1
            self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate_long_range(batch):
    lengths = torch.tensor([len(x) for x, _ in batch], dtype=torch.long)
    T = int(lengths.max())
    X = torch.zeros(len(batch), T, dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)

    for i, (x, _) in enumerate(batch):
        X[i, :len(x)] = x

    mask = torch.arange(T).unsqueeze(0) < lengths.unsqueeze(1)
    return X, lengths, mask, y


train_ds = LongRangeFirstToken(3000, seed=1)
val_ds = LongRangeFirstToken(800, seed=2)

train_loader = DataLoader(
    train_ds, batch_size=64, shuffle=True, collate_fn=collate_long_range
)
val_loader = DataLoader(
    val_ds, batch_size=128, shuffle=False, collate_fn=collate_long_range
)

X, lengths, mask, y = next(iter(train_loader))

# TODO A1-A3: inspect shapes, verify padding/mask, print two unpadded examples.


## B — Vanilla RNN baseline 

Build

\[
\text{Embedding}\rightarrow\texttt{nn.RNN}\rightarrow\text{Linear}.
\]

Use:

- embedding size `16`;
- hidden size `32`;
- `batch_first=True`;
- `pack_padded_sequence(..., enforce_sorted=False)`;
- final hidden state `h_n[0]`;
- cross-entropy;
- Adam with learning rate `3e-3`;
- global gradient clipping at `1.0`;
- at most **6 epochs**.

For a final-state classifier, `h_n` should correspond to the last **valid** timestep of each sequence, not to padded storage positions.

### Deliverables

- validation accuracy;
- parameter count;
- training curve;
- 3–5 sentence interpretation.

Do **not** conclude that poor accuracy by itself proves vanishing gradients.


In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab=10, emb=16, hidden=32):
        super().__init__()
        # TODO: embedding + nn.RNN + classifier

    def forward(self, X, lengths, mask):
        # TODO:
        # 1. embed X
        # 2. pack variable-length sequences
        # 3. run RNN
        # 4. classify from h_n[0]
        raise NotImplementedError


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for X, lengths, mask, y in loader:
        X, lengths, mask, y = (
            X.to(device), lengths.to(device), mask.to(device), y.to(device)
        )
        logits = model(X, lengths, mask)
        pred = logits.argmax(-1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total


def train_classifier(model, epochs=6, lr=3e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        model.train()
        for X, lengths, mask, y in train_loader:
            X, lengths, mask, y = (
                X.to(device), lengths.to(device), mask.to(device), y.to(device)
            )

            opt.zero_grad()
            logits = model(X, lengths, mask)
            loss = F.cross_entropy(logits, y)
            loss.backward()

            # TODO: clip global gradient norm to 1.0

            opt.step()

        va = accuracy(model, val_loader)
        history.append(va)
        print(f"epoch {epoch+1}: val_acc={va:.3f}")

    return history


# TODO B: instantiate, train, report parameter count and plot validation accuracy.


## C — Gated recurrence: GRU and optionally LSTM

Replace the vanilla RNN with a **GRU** while keeping the rest of the setup unchanged.

If your team has 3 members, also implement the corresponding **LSTM**.

### Compare

- validation accuracy;
- parameter count;
- returned recurrent state(s);
- learning curve.

For GRU:

\[
h_t = z_t\odot h_{t-1} + (1-z_t)\odot\tilde h_t.
\]

For LSTM, keep conceptually separate:

\[
(h_t,c_t).
\]

Explain the empirical difference without writing that GRU/LSTM “solves vanishing gradients”.


In [ ]:
class GatedClassifier(nn.Module):
    def __init__(self, kind="gru", vocab=10, emb=16, hidden=32):
        super().__init__()
        assert kind in {"gru", "lstm"}
        # TODO: embedding + selected recurrent module + classifier
        self.kind = kind

    def forward(self, X, lengths, mask):
        # TODO: packed variable-length recurrent classification
        raise NotImplementedError


# TODO C:
# 1. train GRU
# 2. optional LSTM
# 3. create a small comparison table/dictionary:
#    model, params, best_val_accuracy, recurrent_states


## D — Attention pooling + self-attention order probe 

### D1 — Attention pooling over recurrent outputs

Instead of classifying only from the final hidden state, compute a weighted sum over **all valid recurrent outputs**:

\[
e_t = w^\top\tanh(W h_t),
\]

\[
\alpha_t =
\frac{\exp(e_t)}
{\sum_{j\in\text{valid}}\exp(e_j)},
\qquad
c = \sum_t \alpha_t h_t.
\]

Requirements:

- padded positions must get zero probability mass;
- return logits and attention weights;
- train with the same epoch budget;
- visualize attention for at least **6 validation examples**;
- comment on whether the first token receives high weight.

**Caution:** attention weights are not automatically faithful causal explanations.

### D2 — Self-attention order probe

Without implementing a full Transformer, verify experimentally that scaled dot-product self-attention **without positional information** is permutation equivariant.

Then add a simple positional encoding and repeat the probe.


In [ ]:
class AttentionPoolingClassifier(nn.Module):
    def __init__(self, vocab=10, emb=16, hidden=32):
        super().__init__()
        # TODO:
        # embedding
        # GRU or LSTM encoder
        # score network
        # classifier

    def forward(self, X, lengths, mask, return_attention=False):
        # TODO:
        # recurrent sequence output
        # attention scores
        # mask scores BEFORE softmax
        # context vector
        # logits
        raise NotImplementedError


# TODO D1:
# - train
# - report validation accuracy
# - plot attention weights for >= 6 validation samples


In [ ]:
# D2 — self-attention order probe
N, T, D = 2, 6, 8
Z = torch.randn(N, T, D)

Wq = torch.randn(D, D) / math.sqrt(D)
Wk = torch.randn(D, D) / math.sqrt(D)
Wv = torch.randn(D, D) / math.sqrt(D)

def self_attention(z):
    # TODO: Q, K, V, scaled scores, row-wise softmax, output
    raise NotImplementedError

perm = torch.tensor([3, 0, 5, 1, 4, 2])

# TODO:
# Y = self_attention(Z)
# Y_perm = self_attention(Z[:, perm])
# verify Y_perm ≈ Y[:, perm]
#
# Then add a simple positional encoding to Z and show why the same equivalence
# no longer holds for arbitrary token permutation.


# Time-series thread

The next activities introduce a second kind of sequence problem: **forecasting**.

We use a lightweight synthetic series containing:

- trend;
- periodic structure;
- noise.

This makes nonstationary behavior visible while keeping the experiment reproducible.


In [ ]:
# Generate a reproducible time series
n = 1800
t = np.arange(n)

trend = 0.0025 * t
seasonal = (
    1.2 * np.sin(2 * np.pi * t / 60)
    + 0.45 * np.sin(2 * np.pi * t / 15)
)
noise = 0.25 * np.random.randn(n)

series = trend + seasonal + noise

split = int(0.8 * n)
train_series = series[:split]
test_series = series[split:]

plt.figure(figsize=(12, 3))
plt.plot(series)
plt.axvline(split, linestyle="--")
plt.title("Synthetic time series: trend + seasonality + noise")
plt.show()


## E — Explore stationarity before training 

A weakly stationary process has, informally, stable first and second moments over time:

\[
\mathbb E[X_t] = \mu,
\qquad
\operatorname{Var}(X_t)=\sigma^2,
\]

and autocovariance depending on lag rather than absolute time.

Use stationarity as a **diagnostic concept**, not as a compulsory preprocessing rule.

### E1 — Visual diagnostics

For the **training portion only**:

1. plot the series;
2. plot rolling mean and rolling standard deviation;
3. inspect whether level/scale appear stable;
4. plot the first difference

\[
\Delta x_t = x_t-x_{t-1}.
\]

### E2 — ADF test

Run the Augmented Dickey–Fuller test on:

- original training series;
- first-differenced training series.

Report:

- ADF statistic;
- p-value;
- your interpretation.

**Important:** `p < 0.05` is not a proof that every notion of stationarity holds.

### E3 — Data leakage

Explain why rolling statistics, scaling parameters and transformations used for forecasting must be estimated from the **training period**, not from future test observations.


In [ ]:
# E1 — rolling diagnostics
window = 60

# TODO:
# compute rolling mean/std without using future test information
# plot original training series + rolling mean + rolling std

diff_train = np.diff(train_series)

# TODO:
# plot differenced training series


In [ ]:
# E2 — Augmented Dickey-Fuller test
from statsmodels.tsa.stattools import adfuller

def adf_report(x, name):
    result = adfuller(x)
    print(f"{name}")
    print(f"  ADF statistic: {result[0]:.4f}")
    print(f"  p-value:       {result[1]:.4g}")

# TODO:
# adf_report(train_series, "original")
# adf_report(diff_train, "first difference")
#
# Add your interpretation in a markdown cell.


## F — Many-to-many next-step forecasting 

Formulate forecasting as an **aligned many-to-many** problem.

For a window

\[
[x_t,\ldots,x_{t+L-1}],
\]

the target is

\[
[x_{t+1},\ldots,x_{t+L}].
\]

Thus every recurrent output contributes to the training loss.

Use:

\[
h_{1:T}=\operatorname{GRU}(x_{1:T}),
\]

followed by one shared linear map:

\[
\hat x_{t+1}=Wh_t+b.
\]

### PyTorch and “TimeDistributed”

If

```python
h.shape == (N, T, H)
```

then

```python
nn.Linear(H, 1)(h)
```

returns

```python
(N, T, 1)
```

using the **same linear parameters at every timestep**. No special `TimeDistributed` wrapper is needed.

### Tasks

1. create sliding windows from the training period only;
2. fit scaling parameters on the training data only;
3. implement `GRU -> Linear`;
4. train with MSE;
5. plot ground truth vs **one-step predictions**;
6. report test MAE and RMSE.

Do this first on the **original series**.


In [ ]:
def make_windows(values, window=40):
    # TODO:
    # X_i = values[i : i+window]
    # y_i = values[i+1 : i+window+1]
    # return tensors with shape (N, T, 1)
    raise NotImplementedError


class NextStepGRU(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, h0=None):
        # TODO:
        # recurrent output for every timestep
        # same Linear head on every timestep
        raise NotImplementedError


# TODO F:
# - standardize using TRAIN mean/std only
# - build train/test windows without crossing the temporal split
# - train for a modest number of epochs
# - plot one-step predictions
# - compute MAE and RMSE


## G — Recursive multi-step forecasting

One-step training and free-running forecasting are not the same computation.

### During one-step training

The input at the next position is the **true observed value**:

\[
x_t \rightarrow \hat x_{t+1},
\qquad
x_{t+1} \rightarrow \hat x_{t+2}.
\]

### During recursive forecasting

After the context window, future true values are unavailable:

\[
x_T
\rightarrow \hat x_{T+1}
\rightarrow \hat x_{T+2}
\rightarrow \cdots.
\]

The model is therefore exposed to its **own prediction errors**.

### Tasks

1. choose at least **5 forecast origins** in the test period;
2. forecast horizons `1`, `5`, `10`, and `20`;
3. compare recursive forecasts with ground truth;
4. report MAE as a function of horizon;
5. explain error accumulation / distribution shift;
6. repeat the forecasting experiment on the **first-differenced series**;
7. reconstruct the level forecast before comparing with the original series.

### Question

Did differencing help? Discuss both benefits and information loss.  
Do **not** claim that RNNs require stationarity.


In [ ]:
@torch.no_grad()
def recursive_forecast(model, context, horizon):
    # context: tensor shaped (1, T, 1), already transformed/scaled
    # returns 'horizon' future predictions
    model.eval()

    # TODO:
    # 1. run context through recurrent layer to obtain hidden state
    # 2. use last observed value as first one-step input
    # 3. feed each prediction back as the next input
    # 4. collect 'horizon' predictions
    raise NotImplementedError


# TODO G:
# - evaluate several forecast origins
# - horizons = [1, 5, 10, 20]
# - plot at least 3 recursive forecasts
# - compare original-series vs differenced-series pipeline


## H — Final comparison and interpretation 

Create **two compact result tables**.

### Table 1 — Sequence classification

Include:

- Vanilla RNN;
- GRU;
- optional LSTM;
- recurrent encoder + attention pooling.

Columns:

- parameter count;
- best validation accuracy;
- recurrent state(s);
- representation used for classification (`h_n` or attention context).

### Table 2 — Forecasting

Include:

- original vs differenced training representation;
- one-step MAE/RMSE;
- recursive MAE at horizons 1, 5, 10, 20.

Then answer each question in **2–5 sentences**:

1. What evidence supports a difference between vanilla and gated recurrence?
2. Why does gradient clipping control exploding updates but not guarantee long-range learning?
3. Why did `pack_padded_sequence` matter for final-state classification?
4. What exactly did masking change in the attention softmax?
5. Did attention weights behave like faithful explanations? Give one concrete observation.
6. What did the self-attention permutation experiment demonstrate?
7. What evidence suggested that the forecasting series was or was not stationary?
8. Did differencing improve one-step prediction, recursive forecasting, both, or neither?
9. Why is one-step prediction usually easier than long recursive rollout?
10. In what sense does `nn.Linear` on `(N,T,H)` implement the same idea as a time-distributed output layer?

## Contribution statement

List every team member and their contribution. Every member should be able to explain all submitted code and conclusions.


In [ ]:
# TODO H — Build the two required result tables from your experiments:
# 1. sequence classification table
# 2. forecasting table with one-step and recursive metrics
# Display both tables and use them to support the written answers above.


## Optional extension — only after the required work

Choose at most **one**:

- plot forecasting error versus horizon up to 50 steps;
- compare RNN, GRU and LSTM on the forecasting task;
- add a seasonal difference and justify the lag;
- compare final-state classification with masked mean pooling;
- vary sequence length and plot classification accuracy versus dependency distance.
